# Week 16: Advanced MLOps & Distributed Deep Learning
## Notebook 05: Feature Stores & Point-in-Time Correctness from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In production ML systems, the most common source of catastrophic model failure is **Data Leakage (Lookahead Bias)** between training and serving.

Consider a fraud model predicting transaction risk at 2:00 PM:
- If your training query joins the user's total daily spending at 11:59 PM, the model learns from future information that wasn't available at 2:00 PM!
- In production, that future feature is missing, and model performance collapses.

A **Feature Store (e.g. Feast, Tecton)** solves this via **Point-in-Time (As-Of) Joins**:
- **Offline Store (Parquet/Snowflake):** Historical feature join guaranteed to query feature states strictly *prior* to each observation timestamp.
- **Online Store (Redis/DynamoDB):** Sub-10ms key-value lookup for real-time serving.

---

### 2. Mathematical Foundations: Point-in-Time Join
Given:
- Observation table with entity $E_i$ and event timestamp $t_{obs}$.
- Feature table with entity updates $(E_i, t_{feat}, F_k)$.

The as-of join condition requires:
$$F^*(E_i, t_{obs}) = \arg\max_{t_{feat} \le t_{obs}} t_{feat}$$
Features occurring at $t_{feat} > t_{obs}$ are strictly forbidden.

In [ ]:
from datetime import datetime, timedelta
import pandas as pd

# Step 1: Historical Feature Updates Log (User balance updates)
feature_updates = pd.DataFrame([
    {"user_id": 101, "timestamp": datetime(2026, 10, 1, 8, 0), "credit_score": 650, "failed_logins": 0},
    {"user_id": 101, "timestamp": datetime(2026, 10, 1, 13, 0), "credit_score": 620, "failed_logins": 3},
    {"user_id": 101, "timestamp": datetime(2026, 10, 1, 19, 0), "credit_score": 580, "failed_logins": 5},
    {"user_id": 102, "timestamp": datetime(2026, 10, 1, 10, 0), "credit_score": 720, "failed_logins": 0},
    {"user_id": 102, "timestamp": datetime(2026, 10, 1, 15, 0), "credit_score": 715, "failed_logins": 1},
]).sort_values("timestamp")

# Step 2: Training Observation Events (Loan applications / transactions)
observations = pd.DataFrame([
    {"transaction_id": "T1", "user_id": 101, "timestamp": datetime(2026, 10, 1, 14, 0), "is_fraud": 1},
    {"transaction_id": "T2", "user_id": 101, "timestamp": datetime(2026, 10, 1, 9, 0),  "is_fraud": 0},
    {"transaction_id": "T3", "user_id": 102, "timestamp": datetime(2026, 10, 1, 12, 0), "is_fraud": 0},
]).sort_values("timestamp")

print("--- Feature Updates Table ---")
print(feature_updates)

print("\n--- Observation Events Table ---")
print(observations)

# Step 3: Implement Point-in-Time (As-Of) Join
# Merges closest previous feature record where feature_timestamp <= observation_timestamp
point_in_time_dataset = pd.merge_asof(
    observations,
    feature_updates,
    on="timestamp",
    by="user_id",
    direction="backward"
)

print("\n--- Training Dataset with Guaranteed Point-in-Time Correctness ---")
print(point_in_time_dataset[["transaction_id", "user_id", "timestamp", "credit_score", "failed_logins", "is_fraud"]])

# Verification: Transaction T1 occurred at 14:00
# User 101 had updates at 08:00 (score 650, failed 0), 13:00 (score 620, failed 3), and 19:00 (score 580, failed 5)
# T1 correctly joined the 13:00 update (score 620, failed 3) and IGNORED the future 19:00 update!

### 3. Step-by-Step Code Walkthrough
1. **Sorted Timestamps:** `pd.merge_asof` requires timestamps to be sorted.
2. **Direction='backward':** Ensures only records with $t_{feature} \le t_{obs}$ are joined.
3. **Leakage Prevention:** Transaction T1 (14:00) joined the 13:00 record ($score=620, failed=3$) and strictly excluded the future 19:00 record ($score=580$).

---

### 4. Industry Applications & Trade-offs
- **Dual-Store Architecture:** High-throughput batch queries in Snowflake for training; ultra-low latency Redis lookups at inference time.
- **Feature Consistency:** Eliminates training-serving skew by sharing exact transformation definitions across training and production inference.

---

### 5. Top Interview Questions & Answers
**Q1: What is training-serving skew in machine learning?**
> **Answer:** Training-serving skew occurs when the distribution of features or the code used to generate features differs between the offline training pipeline (e.g. SQL batch queries) and the real-time production inference service (e.g. Python microservice). Feature stores provide a single source of truth to guarantee identical logic.

**Q2: What is the purpose of an entity key and timestamp in feature stores?**
> **Answer:** The entity key identifies the domain object (e.g., `user_id`, `merchant_id`), while the event timestamp tracks when the feature value actually materialized in the world, allowing time-travel joins for accurate model backtesting.